# <center> Imagine the Impossible </center>


<font color='red'>**Important**: </font>
- **If the kernel doesn't attach automatically, or if you need to restart the kernel, please choose 'conda_python3' kernel.**
- Run each block of code one by one (starting from the top) either by clicking on "Run" on the menu above or by pressing SHIFT+ENTER on your keyboard.
- For the blocks that you need to complete, you see **"YOUR TURN"** in the title.
- Don't alter the code in places without ``YOUR TURN`` in the title. Just run them. 
- To deploy the pre-trained or fine-tuned model, choose `ml.g4dn.2xlarge` instance type available in your account.
- Have fun!

## Introduction

***
Welcome to this challenge! You are working as a Data Scientist for XYZ advertising startup. Your company’s objective is to leverage the latest advancement in the field of Machine Learning to revolutionize advertising by automatically creating novel product images for wide variety of customers. In this challenge, first you are going to select and deploy a stable diffusion generative AI model on Amazon SageMaker. Then you learn how to use text prompts to create novel images. By the completing this challenge, you will learn how to leverage Amazon SageMaker for creative content generation.


For solving this challenge, we use Amazon [SageMaker JumpStart](https://docs.aws.amazon.com/sagemaker/latest/dg/studio-jumpstart.html)! You can use JumpStart to solve many Machine Learning tasks through one-click in SageMaker Studio, or through [SageMaker JumpStart API](https://sagemaker.readthedocs.io/en/stable/overview.html#use-prebuilt-models-with-sagemaker-jumpstart).  In this challenge, you learn how to use the JumpStart API to generate images from text using state-of-the-art Stable Diffusion models. Furthermore, you will learn how to fine-tune the model to your dataset.

Stable Diffusion is a text-to-image model that enables you to create photorealistic images from just a text prompt. A diffusion model trains by learning to remove noise that was added to a real image. This de-noising process generates a realistic image. These models can also generate images from text alone by conditioning the generation process on the text. For instance, Stable Diffusion is a latent diffusion where the model learns to recognize shapes in a pure noise image and gradually brings these shapes into focus if the shapes match the words in the input text.

Training and deploying large models and running inference on models such as Stable Diffusion is often challenging and include issues such as cuda out of memory, payload size limit exceeded and so on.  JumpStart simplifies this process by providing ready-to-use scripts that have been robustly tested. Furthermore, it provides guidance on each step of the process including the recommended instance types, how to select parameters to guide image generation process, prompt engineering etc. Moreover, you can deploy and run inference on any of the 80+ Diffusion models from JumpStart without having to write any piece of your own code.

In the first part of this notebook, you will learn how to use JumpStart to generate highly realistic and artistic images of any subject, object, environment or scene. This may be as simple as an image of a cute dog or as detailed as a hyper-realistic image of a beautifully decorated cozy kitchen in the style of Greg Rutkowski with dramatic sunset lighting and long shadows with cinematic atmosphere. This can be used to design products and build catalogs for e-commerce business needs or to generate realistic art pieces or stock images.

In the second part of this notebook, you will learn how to use JumpStart to fine-tune the Stable Diffusion model to your dataset. This can be useful when creating art, logos, custom designs, NFTs, and so on, or fun stuff such as generating custom AI images of your pets or avatars of yourself.


Model license: By using this model, you agree to the [CreativeML Open RAIL-M++ license](https://huggingface.co/stabilityai/stable-diffusion-2/blob/main/LICENSE-MODEL).

***

## Overview
Here is what you are going to do in this challenge:

* [Set Up](#Set-Up)
* [Run inference on the pre-trained model](#Run-inference-on-the-pre-trained-model)
    * [Task 1: Select a stable diffusion Model](#Select-Model)
* [Retrieve JumpStart Artifacts & Deploy an Endpoint](#Retrieve-JumpStart-Artifacts-&-Deploy-an-Endpoint)
    * [Task 2 - Deploy the model to an endpoint](#Deploy-Model)
* [Prompt Engineering](#Prompt-Engineering)
    * [Task 3 - Write a prompt](#Task3-Prompt-Engineering)
* [Negative Prompting](#Negative-Prompting)
    * [Task4-negative-prompt-engineering](#Task4-Negative-Prompting)
* Clean up the endpoint
    * [Task5-Clean-Up](#Task5-Clean-Up)


<a id='Set-Up'> </a>

### Set Up

***

Before executing the notebook, there are some initial steps required for set up. This notebook requires ipywidgets and latest version of sagemaker. Don't worry if you get the `ERROR: pip's dependency resolver does not currently take into account all the packages` error.


***

In [ ]:
!pip install nest-asyncio==1.5.5 --quiet
!pip install pandas==1.3.4 --quiet
!pip install ipywidgets==8.1.1 --quiet
!pip install --upgrade sagemaker
!jupyter nbextension enable --py widgetsnbextension

#### Permissions and environment variables

***
To host on Amazon SageMaker, we need to set up and authenticate the use of AWS services. Here, we use the execution role associated with the current notebook as the AWS account role with SageMaker access.

***

In [ ]:
import boto3
import os
import time
import json
import pprint
import numpy as np
import matplotlib.pyplot as plt
import sagemaker
from sagemaker import get_execution_role

In [ ]:
aws_role = get_execution_role()
aws_region = boto3.Session().region_name
sess = sagemaker.Session()
boto_session = boto3.Session()
REGION = boto_session.region_name
print(f'Your region is: {REGION}')
sm_client = boto_session.client(service_name="sagemaker", region_name=REGION) #SageMaker Client
s3_client = boto_session.client(service_name="s3", region_name=REGION) #s3 client

In [ ]:
s3_buckets = s3_client.list_buckets()
for bucket in s3_buckets['Buckets']:
    if 'imagine' in bucket['Name'].lower():
        BUCKET_NAME = bucket['Name']
        break
print(f'The S3 bucket associated with this challenge is: {BUCKET_NAME}')

### Define Helper Functions (please don't modify)

In [ ]:
def submit_your_answer(task_file_name, task_response):
    '''Upload your response to S3 in json format. This acts as your submission!
       Arguments:
           task_file_name (str): name of the task file you are submitting 
           task_response (Dictionary): Response dictionary 
    ''' 
    task_json = json.dumps(task_response)
    with open(task_file_name, 'w') as f:
        f.write(task_json)
    s3_client.upload_file(Bucket=BUCKET_NAME, Filename=os.path.join('./', task_file_name), Key=task_file_name)


def query(model_predictor, text):
    """Query the model predictor."""

    encoded_text = text.encode("utf-8")

    query_response = model_predictor.predict(
        encoded_text,
        {
            "ContentType": "application/x-text",
            "Accept": "application/json",
        },
    )
    return query_response


def parse_response(query_response):
    """Parse response and return generated image and the prompt"""

    response_dict = json.loads(query_response)
    return response_dict["generated_image"], response_dict["prompt"]


def display_img_and_prompt(img, prmpt, submit=False, task_file_name="test.png"):
    """Display hallucinated image."""
    plt.figure(figsize=(12, 12))
    plt.imshow(np.array(img))
    plt.axis("off")
    plt.title(prmpt)
    if submit:
        plt.savefig(task_file_name)
        s3_client.upload_file(Bucket=BUCKET_NAME, Filename=os.path.join('./', task_file_name), Key=task_file_name)

    plt.show()
    
def query_endpoint_with_json_payload(model_predictor, payload, content_type, accept):
    """Query the model predictor with json payload."""

    encoded_payload = json.dumps(payload).encode("utf-8")

    query_response = model_predictor.predict(
        encoded_payload,
        {
            "ContentType": content_type,
            "Accept": accept,
        },
    )
    return query_response


def parse_response_multiple_images(query_response):
    """Parse response and return generated image and the prompt"""

    response_dict = json.loads(query_response)
    return response_dict["generated_images"], response_dict["prompt"]

    

<a id='Run-inference-on-the-pre-trained-model'> </a>
## Run inference on the pre-trained model

***

Using JumpStart, we can perform inference on the pre-trained model, even without fine-tuning it first on a new dataset.
***

<a id='Select-Model'> </a>
### <font color='17C822'> Task 1: Select a Text-to-Image Model </font>
***
Run the following block to get a dropdown menu of an impressive list of GenAI models you can select! Please select <font color='light magenta'>model-txt2img-stabilityai-stable-diffusion-v2-1-base</font> (version 1.1.3) for this challenge. In the following cell you can submit your response. A complete list of SageMaker pre-trained models can also be accessed at [Sagemaker pre-trained Models](https://sagemaker.readthedocs.io/en/stable/doc_utils/pretrainedmodels.html#).

***

In [ ]:
from ipywidgets import Dropdown


# from ipywidgets import Dropdown
from sagemaker.jumpstart.notebook_utils import list_jumpstart_models

# Retrieves all Text-to-Image generation models.
filter_value = "task == txt2img"
txt2img_models = list_jumpstart_models(filter=filter_value)

# display the model-ids in a dropdown to select a model for inference.
model_dropdown = Dropdown(
    options=txt2img_models,
    description="Select a model",
    style={"description_width": "initial"},
    layout={"width": "max-content"},
)
display(model_dropdown)

In [ ]:
model_id, model_version = model_dropdown.value, "1.1.3"
print(model_id, model_version)

##################################################################################################################
### SUBMISSION (DON'T CHANGE THE CODE)
Run the following cell to submit your response. <font color='red'> **DON'T CHANGE THIS LINE** </font>. Just run it as it is once you have selected the model from the dropdown menu.This is your submission for Task #1. You'll receive credit shortly if you answer is correct.
##################################################################################################################

In [ ]:
# DON'T CHANGE THIS LINE OR YOUR SUBMISSION FAILS
submit_your_answer('task1.json', {'model_id': model_dropdown.value})

<a id='Retrieve-JumpStart-Artifacts-&-Deploy-an-Endpoint'> </a>

###  <font> Retrieve JumpStart Artifacts & Deploy an Endpoint </font>

***

Using JumpStart, we can perform inference on the pre-trained model, even without fine-tuning it first on a new dataset. We start by retrieving the `deploy_image_uri`, and `model_uri` for the pre-trained model. To host the pre-trained model, we create an instance of [sagemaker.model.Model](https://sagemaker.readthedocs.io/en/stable/api/inference/model.html) and deploy it. Please note that you may get errors that are not impacting your work.


***

In [ ]:
from sagemaker import image_uris, model_uris, script_uris, hyperparameters, instance_types
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker.utils import name_from_base


endpoint_name = name_from_base(f"imagine-the-impossible-{model_id}")

# Retrieve the inference docker container uri. This is the base HuggingFace container image for the default model above.
deploy_image_uri = image_uris.retrieve(
    region=None,
    framework=None,  # automatically inferred from model_id
    image_scope="inference",
    model_id=model_id,
    model_version=model_version,
    instance_type="ml.g4dn.2xlarge",
)

# Retrieve the model uri. This includes the pre-trained model and parameters as well as the inference scripts.
# This includes all dependencies and scripts for model loading, inference handling etc..
model_uri = model_uris.retrieve(
    model_id=model_id, model_version=model_version, model_scope="inference"
)

# To increase the maximum response size (in bytes) from the endpoint.
env = {
    "MMS_MAX_RESPONSE_SIZE": "20000000",
}

# Create the SageMaker model instance
model = Model(
    image_uri=deploy_image_uri,
    model_data=model_uri,
    role=aws_role,
    predictor_cls=Predictor,
    name=endpoint_name,
    env=env,
)



<a id='Deploy-Model'> </a>
### <font color='17C822'> YOUR TURN: Task 2 - Deploy the model to an endpoint </font>
Complete the arguments for the following line to deploy the model to an endpoint 
You can deploy the trained model by using the ``deploy`` method in your ``model``. The only thing you need to do is to add just three arguments to the ``deploy`` method:
* ``endpoint_name`` (which was defined above)
* ``initial_instance_count`` (which should be 1)
* ``instance_type`` (which should be ``ml.g4dn.2xlarge``)

Complete the ``deploy_parameters`` dictionary below to reflect the above criteria. You can refer to [this page](https://sagemaker.readthedocs.io/en/stable/api/training/estimators.html#sagemaker.estimator.EstimatorBase.deploy) for more information. 

In [ ]:
deploy_parameters = {"endpoint_name":endpoint_name,"initial_instance_count":1,"instance_type":"ml.g4dn.2xlarge"}

##################################################################################################################
### SUBMISSION (DON'T CHANGE THE CODE)
Run the following cell to submit your response. <font color='red'> **DON'T CHANGE THIS LINE** </font>. Just run it as it is once you have completed the ``deploy_parameter`` dictionary above. This is your submission for Task #2. You'll receive credit shortly if you answer is correct.
##################################################################################################################

In [ ]:
# DON'T CHANGE THIS LINE OR YOUR SUBMISSION FAILS
submit_your_answer('task2.json', deploy_parameters)

## Deploy the model
Deploy the model by running the following cell.
### This may take up to 10 minutes. Please do not kill the kernel while you wait.

While you wait, you can check out the [Generate images from text with the stable diffusion model on Amazon SageMaker JumpStart](https://aws.amazon.com/blogs/machine-learning/generate-images-from-text-with-the-stable-diffusion-model-on-amazon-sagemaker-jumpstart/) blog to learn more about Stable Diffusion model and JumpStart.


In [ ]:
model_predictor = model.deploy(predictor_cls= Predictor, **deploy_parameters)

***
Below, we put in some example input text. You can provide any text and the model predicts the image corresponding to that text.

***

In [ ]:
text = "cottage in impressionist style"
query_response = query(model_predictor, text)
img, prmpt = parse_response(query_response)
display_img_and_prompt(img, prmpt)

Impressive, isn't it! Feel free to play with the prompt or continue for more exciting stuff! Hopefully you see how easy it is to deploy a GenAI model on AWS! 

### Supported Inference parameters

***
This model also supports many advanced parameters while performing inference. They include:

* **prompt**: prompt to guide the image generation. Must be specified and can be a string or a list of strings.
* **width**: width of the hallucinated image. If specified, it must be a positive integer divisible by 8.
* **height**: height of the hallucinated image. If specified, it must be a positive integer divisible by 8.
* **num_inference_steps**: Number of denoising steps during image generation. More steps lead to higher quality image. If specified, it must a positive integer.
* **guidance_scale**: Higher guidance scale results in image closely related to the prompt, at the expense of image quality. If specified, it must be a float. guidance_scale<=1 is ignored.
* **negative_prompt**: guide image generation against this prompt. If specified, it must be a string or a list of strings and used with guidance_scale. If guidance_scale is disabled, this is also disabled. Moreover, if prompt is a list of strings then negative_prompt must also be a list of strings. 
* **num_images_per_prompt**: number of images returned per prompt. If specified it must be a positive integer. 
* **seed**: Fix the randomized state for reproducibility. If specified, it must be an integer.

***

In [ ]:
import json

# Training data for different models had different image sizes and it is often observed that the model performs best when the generated image
# has dimensions same as the training data dimension. For dimensions not matching the default dimensions, it may result in a black image.
# Stable Diffusion v1-4 was trained on 512x512 images and Stable Diffusion v2 was trained on 768x768 images.
payload = {
    "prompt": "astronaut on a horse",
    "width": 512,
    "height": 512,
    "num_images_per_prompt": 1,
    "num_inference_steps": 50,
    "guidance_scale": 7.5,
    "seed": 1,
}


query_response = query_endpoint_with_json_payload(
    model_predictor, payload, "application/json", "application/json"
)
generated_images, prompt = parse_response_multiple_images(query_response)

for img in generated_images:
    display_img_and_prompt(img, prompt)

### Compressed Image Output

---

Default response type above from an endpoint is a nested array with RGB values and if the generated image size is large, this may hit response size limit. To address this, we also support endpoint response where each image is returned as a JPEG image returned as bytes. To do this, please set `Accept = 'application/json;jpeg'`.


---

In [ ]:
from PIL import Image
from io import BytesIO
import base64
import json


def display_encoded_images(generated_images, title, submit=False, task_name="test.png"):
    """Decode the images and convert to RGB format and display

    Args:
    generated_images: are a list of jpeg images as bytes with b64 encoding.
    """

    for generated_image in generated_images:
        generated_image_decoded = BytesIO(base64.b64decode(generated_image.encode()))
        generated_image_rgb = Image.open(generated_image_decoded).convert("RGB")
        display_img_and_prompt(generated_image_rgb, title, submit, task_name)


def compressed_output_query_and_display(payload, title, submit=False, task_name="test.png"):
    query_response = query_endpoint_with_json_payload(
        model_predictor, payload, "application/json", "application/json;jpeg"
    )
    generated_images, prompt = parse_response_multiple_images(query_response)

    display_encoded_images(generated_images, title, submit, task_name)


payload = {
    "prompt": "astronaut on a horse",
    "width": 512,
    "height": 512,
    "num_images_per_prompt": 1,
    "num_inference_steps": 50,
    "guidance_scale": 7.5,
    "seed": 1,
}
compressed_output_query_and_display(payload, "generated image with compressed response type", submit=False, task_name="test.png")

<a id='Prompt-Engineering'> </a>
### Prompt Engineering
---
Writing a good prompt can sometime be an art. It is often difficult to predict whether a certain prompt will yield a satisfactory image with a given model. However, there are certain templates that have been observed to work. Broadly, a prompt can be roughly broken down into three pieces: (i) type of image (photograph/sketch/painting etc.), (ii) description (subject/object/environment/scene etc.) and (iii) the style of the image (realistic/artistic/type of art etc.). You can change each of the three parts individually to generate variations of an image. Adjectives have been known to play a significant role in the image generation process. Also, adding more details help in the generation process.

To generate a realistic image, you can use phrases such as “a photo of”, “a photograph of”, “realistic” or “hyper realistic”. To generate images by artists you can use phrases like “by Pablo   Piccaso” or “oil painting by Rembrandt” or “landscape art by Frederic Edwin Church” or “pencil drawing by Albrecht Dürer”. You can also combine different artists as well. To generate artistic image by category, you can add the art category in the prompt such as “lion on a beach, abstract”. Some other categories include “oil painting”, “pencil drawing, “pop art”, “digital art”, “anime”, “cartoon”, “futurism”, “watercolor”, “manga” etc. You can also include details such as lighting or camera lens such as 35mm wide lens or 85mm wide lens and details about the framing (portrait/landscape/close up etc.).

Note that model generates different images even if same prompt is given multiple times. So, you can generate multiple images and select the image that suits your application best.

---

In [ ]:
prompts = [
    "An african woman with turban smiles at the camera, pexels contest winner, smiling young woman, face is wrapped in black scarf, a cute young woman, loosely cropped, beautiful girl, acting headshot",
    "Ancient Japanese Samurai, a person standing on a ledge in a city at night, cyberpunk art, trending on shutterstock, batman mecha, stylized cyberpunk minotaur logo, cinematic, cyberpunk",
    "Character design of a robot warrior, concept art, contest winner, diverse medical cybersuits, Football armor, triade color scheme, black shirt underneath armor, in golden armor, clothes in military armor, high resolution render, octane",
    "A croissant sitting on top of a yellow plate, a portait, trending on unsplash, sitting on a mocha-coloured table, magazine, woodfired, bakery, great composition, amber",
    "symmetry!! portrait of vanessa hudgens in the style of horizon zero dawn, machine face, intricate, elegant, highly detailed, digital painting, artstation, concept art, smooth, sharp focus, illustration, art by artgerm and greg rutkowski and alphonse mucha, 8 k",
    "landscape of the beautiful city of paris rebuilt near the pacific ocean in sunny california, amazing weather, sandy beach, palm trees, splendid haussmann architecture, digital painting, highly detailed, intricate, without duplication, art by craig mullins, greg rutkwowski, concept art, matte painting, trending on artstation",
]
for prompt in prompts:
    payload = {"prompt": prompt, "width": 512, "height": 512, "seed": 1}
    compressed_output_query_and_display(payload, "generated image with detailed prompt")

<a id='Task3-Prompt-Engineering'> </a>
### <font color='17C822'> YOUR TURN: Task 3 - Write a prompt </font>
Write a prompt to generate a photorealistic headshot a smiling old woman with glasses sitting under a tree with a medium length hair! Experiment with your prompt below and once ready, change the ``submit`` argument from ``False`` to  ``True`` in the ``compressed_output_query_and_display`` function. Don't change anything else!

In [ ]:
task3_prompt = "A happy old woman with glasses sitting under a tree, photographic style, headshot, high resolution, medium length hair",

#Please change submit value to True once happy with your result for submission!
payload = {"prompt": task3_prompt, "width": 512, "height": 512, "seed": 1}
compressed_output_query_and_display(payload, "generated image with detailed prompt", submit=True, task_name='task3.png')

<a id='Negative-Prompting'> </a>
### Negative Prompting
---

Negative prompt is an important parameter while generating images using Stable Diffusion Models. It provides you an additional control over the image generation process and let you direct the model to avoid certain objects, colors, styles, attributes and more from the generated images.  

---

In [ ]:
prompt = "emma watson as nature magic celestial, top down pose, long hair, soft pink and white transparent cloth, space, D&D, shiny background, intricate, elegant, highly detailed, digital painting, artstation, concept art, smooth, sharp focus, illustration, artgerm, bouguereau"
payload = {"prompt": prompt, "seed": 0}
compressed_output_query_and_display(payload, "generated image with no negative prompt")


negative_prompt = "windy"
payload = {"prompt": prompt, "negative_prompt": negative_prompt, "seed": 0}
compressed_output_query_and_display(
    payload, f"generated image with negative prompt: `{negative_prompt}`"
)

---

Even though, you can specify many of these concepts in the original prompt by specifying negative words “without”, “except”, “no” and “not”, Stable Diffusion models have been observed to not understand the negative words very well. Thus, you should use negative prompt parameter when tailoring the image to your use case. 

---

<a id='Task4-Negative-Prompting'> </a>
### <font color='17C822'> YOUR TURN: Task 4 - Negative Prompt </font>
First, let's try to create a portrait of a man without beard:

In [ ]:
prompt = "a portrait of a man without beard"
payload = {"prompt": prompt, "seed": 0}
compressed_output_query_and_display(payload, f"prompt: `{prompt}`, negative prompt: None")

It didn't quite work! As mentioned before, current stable diffusion models are not very good negative words in the prompt. So, let's explicitly provide a negative prompt and see what happens. Please don't change the prompt below, just provide the negative prompt. When ready, change submit to True.

In [ ]:
prompt = "a portrait of a man" #Don't modify!
negative_prompt = "beard"
payload = {"prompt": prompt, "negative_prompt": negative_prompt, "seed": 0} #Don't modify

# Once happy with your response, change the submit to True in the following line
compressed_output_query_and_display(
    payload, f"prompt: `{prompt}`, negative prompt: `{negative_prompt}`"
, submit=True, task_name='task4.png')

---
While trying to generate images, we recommend starting with prompt and progressively building negative prompt to exclude the subjects/styles that you do not want in the image.

---

### Conclusion
---
In this challenge, we learnt how to deploy a pre-trained Stable Diffusion model on SageMaker using JumpStart. We saw that Stable Diffusion models can generate highly photo-realistic images from text.  JumpStart also provides additional 84 diffusion models which have been trained to generate images from different themes and different languages. 

Although creating impressive images can find use in industries ranging from art to NFTs and beyond, today we also expect AI to be personalizable. JumpStart provides fine-tuning capability to the pre-trained models so that you can adapt the model to your own use case with as little as five training images. This can be useful when creating art, logos, custom designs, NFTs, and so on, or fun stuff such as generating custom AI images of your pets or avatars of yourself. To learn more about Stable Diffusion fine-tuning, please check out the blog [Fine-tune text-to-image Stable Diffusion models with Amazon SageMaker JumpStart](https://aws.amazon.com/blogs/machine-learning/fine-tune-text-to-image-stable-diffusion-models-with-amazon-sagemaker-jumpstart/).

<a id='Task5-Clean-Up'> </a>
### <font color='17C822'> YOUR TURN: Task 5 - clean Up </font>
In order to delete the endpoint, you can call ``delete_endpoint`` method from the SageMaker client. Remember, we created the SageMaker client, earlier in this notebook. Note that you need to write just **one line** of code! You receive credit automatically, once the endpoint is deleted. You can see [this page](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/sagemaker.html#SageMaker.Client.delete_endpoint) for more information. Delete the model endpoint by completing to following line:

In [ ]:
# Delete the SageMaker endpoint
sm_client.delete_endpoint(EndpointName = endpoint_name) 

## Congratulations! You have now completed this challenge!